# POC 02 — Incremental API ingestion
Attach `lh_api_orders` as the default Lakehouse. Upload the first two JSON pages before run 1, then add page 3 before run 2.

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window

source_path = 'Files/api_source/*.json'
incoming_raw = spark.read.json(source_path)
incoming = (incoming_raw
    .select(
        F.col('event_id').cast('string'),
        F.col('page_number').cast('int'),
        F.col('order_id').cast('string'),
        F.col('customer_id').cast('string'),
        F.col('status').cast('string'),
        F.col('amount').cast('decimal(12,2)'),
        F.to_timestamp('updated_at').alias('updated_at'))
    .withColumn('_source_file', F.input_file_name())
    .withColumn('_ingested_at', F.current_timestamp())
    .dropDuplicates(['event_id']))

required = ['event_id', 'order_id', 'updated_at']
invalid = incoming.filter(F.expr(' OR '.join([f'{c} IS NULL' for c in required])))
assert invalid.count() == 0, 'Required API fields contain null values'
display(incoming)

## Bronze — idempotent event landing
`event_id` is immutable. MERGE inserts unseen events and ignores pages already processed.

In [ ]:
table_name = 'bronze_api_order_events'
before_count = spark.table(table_name).count() if spark.catalog.tableExists(table_name) else 0

if spark.catalog.tableExists(table_name):
    (DeltaTable.forName(spark, table_name).alias('target')
        .merge(incoming.alias('source'), 'target.event_id = source.event_id')
        .whenNotMatchedInsertAll()
        .execute())
else:
    incoming.write.format('delta').mode('overwrite').saveAsTable(table_name)

bronze = spark.table(table_name)
after_count = bronze.count()
new_events = after_count - before_count
print(f'New events inserted: {new_events}; Bronze events: {after_count}')

## Silver — current state per business key
An order may have many events. Silver keeps only the newest `updated_at` for each `order_id`.

In [ ]:
latest = Window.partitionBy('order_id').orderBy(F.col('updated_at').desc(), F.col('event_id').desc())
silver = (bronze
    .withColumn('_version_rank', F.row_number().over(latest))
    .filter(F.col('_version_rank') == 1)
    .drop('_version_rank'))
silver.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable('silver_api_orders_current')
display(silver.orderBy('order_id'))

## Gold, watermark, and run evidence

In [ ]:
gold = (silver.groupBy('status')
    .agg(F.countDistinct('order_id').alias('orders_count'),
         F.sum('amount').alias('total_amount')))
gold.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable('gold_api_order_summary')

watermark = bronze.agg(F.max('updated_at')).first()[0]
(spark.createDataFrame([(watermark,)], ['last_successful_updated_at'])
    .write.format('delta').mode('overwrite').saveAsTable('control_api_watermark'))

run_log = (spark.range(1).select(
    F.current_timestamp().alias('run_at'),
    F.lit(incoming_raw.count()).alias('source_rows_scanned'),
    F.lit(new_events).alias('new_events_inserted'),
    F.lit(after_count).alias('bronze_events_after')))
run_log.write.format('delta').mode('append').saveAsTable('api_ingestion_run_log')
display(gold.orderBy('status'))